# Solutions · 13 · Symbolic mathematics with SymPy

Worked solutions to the exercises of [notebook 13](../notebooks/13_symbolic_sympy.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engmath is missing): install it from GitHub
    import engmath
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engmath"], check=True)
import numpy as np
import matplotlib.pyplot as plt
import engmath
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")
import sympy as sp

## Exercise 1

Derive the deflection of a *simply supported* beam ($v = v'' = 0$ at both ends) under a uniform load
   and confirm the maximum deflection $5wL^4/(384EI)$.

In [2]:
x, L, EI, w = sp.symbols("x L EI w", positive=True)
v = sp.Function("v")
bcs = {v(0): 0, v(x).diff(x, 2).subs(x, 0): 0, v(L): 0, v(x).diff(x, 2).subs(x, L): 0}
sol = sp.factor(sp.dsolve(sp.Eq(EI*v(x).diff(x, 4), w), v(x), ics=bcs).rhs)
print("v(x) =", sol)
print("midspan deflection =", sp.simplify(sol.subs(x, L/2)))
print("slope at the support =", sp.simplify(sol.diff(x).subs(x, 0)))

v(x) = w*x*(-L + x)*(-L**2 - L*x + x**2)/(24*EI)
midspan deflection = 5*L**4*w/(384*EI)
slope at the support = L**3*w/(24*EI)


The maximum deflection (at midspan, by symmetry) is $5wL^4/(384EI)$ - only $5/48 \approx 0.10$ of the
cantilever's $wL^4/(8EI)$ for the same span and load, because both ends are supported.

## Exercise 2

Derive the leading error term of the forward difference $(f_1 - f_0)/h$ and of the 5-point central
   difference $(f_{-2} - 8f_{-1} + 8f_1 - f_2)/(12h)$. What are their orders?

In [3]:
h = sp.Symbol("h", positive=True)
F = sp.symbols("F0:8")
taylor = lambda s: sum(F[j]*s**j/sp.factorial(j) for j in range(8))
forward = (taylor(h) - taylor(0)) / h - F[1]
five_point = (taylor(-2*h) - 8*taylor(-h) + 8*taylor(h) - taylor(2*h)) / (12*h) - F[1]
print("forward difference error:   ", sp.expand(forward).as_leading_term(h))
print("5-point central error:      ", sp.expand(five_point).as_leading_term(h))

forward difference error:    F2*h/2
5-point central error:       -F5*h**4/30


The forward difference has leading error $h f''/2$: first order. The 5-point central formula cancels all
terms up to $h^4$: its leading error is $-h^4 f^{(5)}/30$, fourth order. More points buy higher order, at the
cost of a wider stencil (awkward near boundaries) and more sensitivity to noise.

## Exercise 3

Use `sp.dsolve` to solve the pin-fin equation of notebook 15, $T'' = m^2 (T - T_\infty)$, with
   $T(0) = T_b$ and an insulated tip $T'(L) = 0$. Check the formula against `bvp.finite_difference`.

In [4]:
x, m, Tb, Tinf, L = sp.symbols("x m T_b T_inf L", positive=True)
T = sp.Function("T")
sol = sp.dsolve(sp.Eq(T(x).diff(x, 2), m**2*(T(x) - Tinf)), T(x),
                ics={T(0): Tb, T(x).diff(x).subs(x, L): 0}).rhs
print("T(x) =", sol)
textbook = Tinf + (Tb - Tinf)*sp.cosh(m*(L - x))/sp.cosh(m*L)
print("equal to the textbook form:", sp.simplify((sol - textbook).rewrite(sp.exp)) == 0)
from engmath import bvp
vals = {m: 10.0, Tb: 100.0, Tinf: 20.0, L: 0.05}
T_fun = sp.lambdify(x, sol.subs(vals), "numpy")
xf, Tf = bvp.finite_difference(lambda s: 0*s, lambda s: -100.0 + 0*s, lambda s: -100.0*20.0 + 0*s, 0, 0.05, 100,
                               ("dirichlet", 100.0), ("robin", 1.0, 0.0, 0.0))
print(f"max |finite differences - SymPy| = {np.max(np.abs(Tf - T_fun(xf))):.1e} K")

T(x) = T_inf + (T_b/(exp(2*L*m) + 1) - T_inf/(exp(2*L*m) + 1))*exp(m*x) + (T_b*exp(2*L*m)/(exp(2*L*m) + 1) - T_inf*exp(2*L*m)/(exp(2*L*m) + 1))*exp(-m*x)


equal to the textbook form: True
max |finite differences - SymPy| = 1.7e-05 K


SymPy returns the solution with exponentials; the symbolic check confirms it equals the textbook form
$T - T_\infty = (T_b - T_\infty)\cosh(m(L - x))/\cosh(mL)$. The finite-difference solution with an insulated
(Neumann) tip agrees to within the expected second-order discretisation error.

## Exercise 4

**Implement it yourself:** use the Taylor-series technique to derive the 3-point formula for the *second* derivative on an uneven grid and its leading error term. Is it still second order? Compare with `differentiate.second_derivative`.

In [5]:
h1, h2, t = sp.symbols("h1 h2 t", positive=True)
f0, f1, f2 = sp.symbols("f0 f1 f2")
F = sp.symbols("F0:6")
second = sp.expand(sp.diff(sp.interpolate([(-h1, f0), (0, f1), (h2, f2)], t), t, 2))
print("f'' ≈", sp.factor(second))
taylor = lambda s: sum(F[j]*s**j/sp.factorial(j) for j in range(6))
err = sp.expand(second.subs({f0: taylor(-h1), f1: taylor(0), f2: taylor(h2)}) - F[2])
print("leading error:", sp.factor(err.coeff(F[3])*F[3]))
print("equal spacing:", sp.factor(err.subs({h1: sp.Symbol("h"), h2: sp.Symbol("h")})))
from engmath import differentiate
xs = np.array([0.0, 0.3, 0.5, 1.0, 1.2]); ys = np.exp(xs)
fun = sp.lambdify((f0, f1, f2, h1, h2), second)
print("formula:", fun(ys[1], ys[2], ys[3], xs[2]-xs[1], xs[3]-xs[2]), " library:", differentiate.second_derivative(xs, ys)[2])

f'' ≈ 2*(f0*h2 - f1*h1 - f1*h2 + f2*h1)/(h1*h2*(h1 + h2))
leading error: -F3*(h1 - h2)/3
equal spacing: F4*h**2/12
formula: 1.8423108568491706  library: 1.8423108568491737


On an uneven grid the second-derivative formula is only **first order**: its leading error is
$(h_2 - h_1)f'''/3$. The first-order term cancels only when $h_1 = h_2$, leaving the familiar second-order
error $h^2 f^{(4)}/12$. Uneven grids must therefore change smoothly (neighbouring spacings nearly equal) to
keep second-order accuracy for second derivatives.